# Step 1: Extract Maker–Application Links from PATSTAT

## Purpose
Filter the PATSTAT persons table (TLS206) to automotive manufacturers of interest, then link each manufacturer to its patent applications via the TLS207 person–application link table.

## Inputs (raw PATSTAT files, all in `patstat_raw/` next to the repository folder; see SETUP.md, Part 2)
| File | Description |
|------|-------------|
| `patstat_raw/tls206_part01.csv` | PATSTAT persons table, part 1 (~114 M rows total) |
| `patstat_raw/tls206_part02.csv` | PATSTAT persons table, part 2 |
| `patstat_raw/tls207_part01.csv` | Person–application links |

## Output
| File | Description |
|------|-------------|
| `maker_app_links.csv` | (person_id, appln_id, psn_name, person_ctry_code) for matched firms — read by **Step 2** as `../step1/maker_app_links.csv` |


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import pandas as pd
import gc          # garbage collector — needed to free the huge TLS206 parts
from pathlib import Path


In [ ]:
# ── File paths ────────────────────────────────────────────────────────────────
# Raw PATSTAT exports live outside the repository (see SETUP.md, Part 2).
# Default: a folder `patstat_raw/` next to the repository folder; set the
# environment variable PATSTAT_DIR to use another location.
import os
PATSTAT_DIR  = Path(os.environ.get('PATSTAT_DIR', '../../../patstat_raw'))
TLS206_PARTS = [PATSTAT_DIR / 'tls206_part01.csv', PATSTAT_DIR / 'tls206_part02.csv']
TLS207_FILE  = PATSTAT_DIR / 'tls207_part01.csv'
_missing = [p for p in TLS206_PARTS + [TLS207_FILE] if not p.exists()]
if _missing:
    for p in _missing:
        print('MISSING', p)
    raise FileNotFoundError(
        f'{len(_missing)} raw PATSTAT file(s) missing. Put ALL PATSTAT exports into '
        f'{PATSTAT_DIR.resolve()} (default: patstat_raw/ next to the repository folder; '
        'see SETUP.md, Part 2).')
OUTPUT_FILE  = 'maker_app_links.csv'   # consumed by Step 2

# Columns to load — loading only what we need reduces peak RAM from ~8 GB to ~2 GB.
COLS_206  = ['person_id', 'psn_name', 'person_ctry_code']
COLS_207  = ['person_id', 'appln_id', 'applt_seq_nr']
DTYPE_206 = {'person_id': 'int32', 'psn_name': 'str', 'person_ctry_code': 'str'}
DTYPE_207 = {'person_id': 'int32', 'appln_id': 'int32', 'applt_seq_nr': 'int16'}

# ── Target maker names ────────────────────────────────────────────────────────
# These are PATSTAT standardised names (psn_name).
# GERMAN_MAKERS form the *treatment group* in the DDD design (Dieselgate firms).
# CONTROL_MAKERS are the comparison group used to identify the counterfactual.
# The list is intentionally broad so that Step 2 can refine it using patent-count
# thresholds; any firm with fewer than 400 applications will be dropped there.
GERMAN_MAKERS = [
    'VOLKSWAGEN', 'AUDI', 'PORSCHE',
    'DAIMLER', 'DAIMLER-BENZ', 'DAIMLERCHRYSLER', 'MERCEDES-BENZ',
    'BMW', 'OPEL',
]
CONTROL_MAKERS = [
    # Japanese (world's largest patent filers in automotive, used as main controls)
    'TOYOTA', 'LEXUS', 'SCION',
    'HONDA', 'ACURA',
    'NISSAN', 'INFINITI',
    'MITSUBISHI', 'MAZDA', 'SUBARU', 'SUZUKI',
    # Korean
    'HYUNDAI', 'GENESIS', 'KIA',
    # American
    'FORD', 'LINCOLN', 'MERCURY',
    'GMC', 'GENERAL MOTORS', 'GENERAL MOTORS COMPANY', 'GM', 'GM GLOBAL', 'CHEVROLET', 'BUICK', 'CADILLAC', 'HUMMER', 'PONTIAC', 'SATURN',
    'CHRYSLER', 'DODGE', 'JEEP', 'RAM',
    # Swedish / British
    'VOLVO', 'JAGUAR', 'LAND ROVER', 'SAAB', 'MINI',
    # Italian
    'FIAT', 'ALFA ROMEO',
    # Pure EV (provides the EV-technology benchmark)
    'TESLA',
]


In [ ]:
def build_regex_pattern(makers: list) -> str:
    """
    Combine maker names into a single OR-regex with word-boundary anchors.

    Why word boundaries (\\b)?
        Without them, 'FORD' would also match 'FORD-WERKE' and 'FORD GLOBAL
        TECHNOLOGIES' — causing unintended partial matches in the broad
        psn_name strings stored in PATSTAT.  \\b ensures each token is
        matched as a standalone word.

    Parameters
    ----------
    makers : list of str
        Standardised PATSTAT names to search for.

    Returns
    -------
    str : regex pattern ready for pandas str.contains(regex=True, case=False)
    """
    return '|'.join(r'\b' + name + r'\b' for name in makers)


In [ ]:
def load_and_filter_persons(filepath: str, regex_pattern: str) -> pd.DataFrame:
    """
    Load one TLS206 part and keep only rows whose psn_name matches target makers.

    Memory strategy: TLS206 has ~114 M rows split across two files.  We load
    each part separately, filter immediately, and release the raw DataFrame
    before loading the next part.  This keeps peak RAM below ~3 GB.

    Parameters
    ----------
    filepath : str
        Path to a tls206_partNN.csv file.
    regex_pattern : str
        Pattern returned by build_regex_pattern().

    Returns
    -------
    pd.DataFrame : filtered rows — typically 100–200 K rows per part.
    """
    df = pd.read_csv(filepath, usecols=COLS_206, dtype=DTYPE_206, on_bad_lines='skip')
    mask = df['psn_name'].str.contains(regex_pattern, case=False, na=False, regex=True)
    filtered = df[mask].copy()
    print(f'  {Path(filepath).name}: {len(df):>12,} rows → {len(filtered):,} matched')
    return filtered


In [ ]:
def load_applicant_links(filepath: str) -> pd.DataFrame:
    """
    Load TLS207 and retain *applicant* records only (applt_seq_nr > 0).

    TLS207 links persons (firms and inventors) to patent applications.
    The column applt_seq_nr distinguishes roles:
      - applt_seq_nr > 0  →  applicant (the legal owner / filing firm)
      - applt_seq_nr == 0 →  inventor  (the natural person who invented)
    We are interested in *institutional filers*, so only applicants are kept.

    Parameters
    ----------
    filepath : str
        Path to tls207_part01.csv.

    Returns
    -------
    pd.DataFrame : columns (person_id, appln_id)
    """
    df = pd.read_csv(filepath, usecols=COLS_207, dtype=DTYPE_207)
    applicants = df[df['applt_seq_nr'] > 0][['person_id', 'appln_id']].copy()
    print(f'  TLS207: {len(df):,} total rows → {len(applicants):,} applicant links')
    return applicants


In [ ]:
def merge_persons_with_applications(persons: pd.DataFrame,
                                    links: pd.DataFrame) -> pd.DataFrame:
    """
    Inner-join filtered persons with applicant links on person_id.

    An inner join means we only keep applications for which we found a
    matching firm in our target list.  Applications filed by individuals or
    by firms outside our target set are discarded here.

    Parameters
    ----------
    persons : pd.DataFrame  — from load_and_filter_persons()
    links   : pd.DataFrame  — from load_applicant_links()

    Returns
    -------
    pd.DataFrame : (person_id, appln_id, psn_name, person_ctry_code)
    """
    merged = pd.merge(links, persons, on='person_id', how='inner')
    print(f'  Merged rows  : {len(merged):,}')
    print(f'  Unique appln_id: {merged["appln_id"].nunique():,}')
    print(f'  Unique psn_name: {merged["psn_name"].nunique():,}')
    return merged


In [ ]:
def main() -> None:
    """
    Full Step 1 pipeline:
      1. Build the combined regex for all target manufacturers.
      2. Load each TLS206 part, filter to target makers, free raw data.
      3. Concatenate filtered parts.
      4. Load applicant-only rows from TLS207.
      5. Inner-join persons with links and save to maker_app_links.csv.
    """
    print('=== Step 1: Extract Maker–Application Links ===')

    # Step 1a — build regex
    all_makers  = GERMAN_MAKERS + CONTROL_MAKERS
    regex_pat   = build_regex_pattern(all_makers)
    print(f'Regex covers {len(all_makers)} maker terms')

    # Step 1b — filter persons table (each part separately to save RAM)
    print('\nFiltering TLS206 person records...')
    parts = []
    for fp in TLS206_PARTS:
        part = load_and_filter_persons(fp, regex_pat)
        parts.append(part)
        gc.collect()

    persons = pd.concat(parts, ignore_index=True)
    del parts
    gc.collect()

    # Step 1c — load applicant links
    print('\nLoading TLS207 applicant links...')
    links = load_applicant_links(TLS207_FILE)

    # Step 1d — merge and save
    print('\nMerging...')
    result = merge_persons_with_applications(persons, links)
    del persons, links
    gc.collect()

    result.to_csv(OUTPUT_FILE, index=False)
    print(f'\n✓ Saved → {OUTPUT_FILE}  ({len(result):,} rows)')


main()
